# Download UN SDG API Data 

In [ ]:
# (1) Paths and settings
from pathlib import Path

PROJECT_ROOT = Path("/Users/karwailin/Desktop/OAwalk")
ISO_PATH = PROJECT_ROOT / "data" / "iso36.csv"
OUT_DIR = PROJECT_ROOT / "data" / "sdg" / "api_iso36_all"
BY_INDICATOR_DIR = OUT_DIR / "by_indicator"

API_BASE = "https://unstats.un.org/SDGAPI/v1/sdg"
PAGE_SIZE = 5000
REQUEST_TIMEOUT = 60
SLEEP_SECONDS = 0.15

# Set to a small integer while testing, e.g. 3. Use None for all indicators.
MAX_INDICATORS = None

# If False, already downloaded per-indicator files are skipped.
OVERWRITE_INDICATOR_FILES = False

OUT_DIR.mkdir(parents=True, exist_ok=True)
BY_INDICATOR_DIR.mkdir(parents=True, exist_ok=True)

print("ISO_PATH:", ISO_PATH)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# (2) Dependencies
import json
import time
from datetime import datetime
from typing import Dict, Iterable, List, Optional

import pandas as pd
import requests

print("OK: imports done")

In [ ]:
# (3) Load ISO36 countries
iso = pd.read_csv(ISO_PATH)
required_cols = {"isocountry", "isocountry_c", "iso3c"}
missing = required_cols - set(iso.columns)
if missing:
    raise ValueError(f"iso36.csv is missing required columns: {sorted(missing)}")

iso = iso.copy()
iso["GeoAreaCode"] = pd.to_numeric(iso["isocountry"], errors="coerce").astype("Int64")
iso["GeoAreaName_local"] = iso["isocountry_c"].astype(str).str.strip()
iso["iso3c"] = iso["iso3c"].astype(str).str.strip().str.upper()
iso = iso.dropna(subset=["GeoAreaCode"]).drop_duplicates("GeoAreaCode").sort_values("GeoAreaCode")
iso["GeoAreaCode_str"] = iso["GeoAreaCode"].astype(int).astype(str)
area_codes = iso["GeoAreaCode_str"].tolist()

iso_out = OUT_DIR / "iso36_sdg_area_codes.csv"
iso.to_csv(iso_out, index=False)

print("Countries:", len(area_codes))
print("Saved:", iso_out)
iso.head()

In [ ]:
# (4) HTTP helpers
session = requests.Session()
session.headers.update({"User-Agent": "OAwalk SDG downloader (research; requests)"})


def get_json(url: str, params: Optional[List[tuple]] = None, max_retries: int = 5) -> object:
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
            if resp.status_code in {429, 500, 502, 503, 504}:
                raise requests.HTTPError(f"HTTP {resp.status_code}: {resp.text[:200]}")
            resp.raise_for_status()
            return resp.json()
        except Exception as exc:
            last_error = exc
            wait = min(60, 2 ** attempt)
            print(f"Request failed attempt {attempt}/{max_retries}; waiting {wait}s; error: {exc}")
            time.sleep(wait)
    raise RuntimeError(f"Request failed after {max_retries} attempts: {last_error}")


def area_code_params(codes: Iterable[str]) -> List[tuple]:
    return [("areaCode", str(code)) for code in codes]

print("OK: HTTP helpers ready")

In [ ]:
# (5) Download indicator and series metadata
indicators_json = get_json(f"{API_BASE}/Indicator/List")
series_json = get_json(f"{API_BASE}/Series/List")

indicators = pd.json_normalize(indicators_json)
series = pd.json_normalize(series_json)

indicator_path = OUT_DIR / "sdg_indicator_metadata.csv"
series_path = OUT_DIR / "sdg_series_metadata.csv"
raw_indicator_json_path = OUT_DIR / "sdg_indicator_metadata_raw.json"
raw_series_json_path = OUT_DIR / "sdg_series_metadata_raw.json"

indicators.to_csv(indicator_path, index=False)
series.to_csv(series_path, index=False)
raw_indicator_json_path.write_text(json.dumps(indicators_json, ensure_ascii=False, indent=2))
raw_series_json_path.write_text(json.dumps(series_json, ensure_ascii=False, indent=2))

indicator_codes = indicators["code"].dropna().astype(str).drop_duplicates().sort_values().tolist()
if MAX_INDICATORS is not None:
    indicator_codes = indicator_codes[:MAX_INDICATORS]

print("Indicators to download:", len(indicator_codes))
print("Saved:", indicator_path)
print("Saved:", series_path)
indicators.head()

In [ ]:
# (6) Flatten SDG data records
def flatten_record(record: Dict) -> Dict:
    out = dict(record)

    for key in ["goal", "target", "indicator", "footnotes"]:
        value = out.get(key)
        if isinstance(value, list):
            out[key] = "|".join(str(v) for v in value)

    attrs = out.pop("attributes", None)
    if isinstance(attrs, dict):
        for k, v in attrs.items():
            out[f"attr_{k}"] = v

    dims = out.pop("dimensions", None)
    if isinstance(dims, dict):
        for k, v in dims.items():
            safe_key = str(k).replace(" ", "_").replace("/", "_")
            out[f"dim_{safe_key}"] = v

    return out


def safe_indicator_filename(indicator_code: str) -> str:
    return indicator_code.replace(".", "_").replace("/", "_") + ".csv"

print("OK: flatten helpers ready")

In [ ]:
# (7) Download all historical data by indicator for ISO36 countries
log_rows = []
error_rows = []

for idx, indicator_code in enumerate(indicator_codes, start=1):
    out_path = BY_INDICATOR_DIR / safe_indicator_filename(indicator_code)
    if out_path.exists() and not OVERWRITE_INDICATOR_FILES:
        log_rows.append({
            "indicator": indicator_code,
            "status": "skipped_existing",
            "rows": pd.read_csv(out_path, nrows=0).shape[0],
            "file": str(out_path),
            "downloaded_at": datetime.now().isoformat(timespec="seconds"),
        })
        print(f"[{idx}/{len(indicator_codes)}] skip existing {indicator_code}")
        continue

    print(f"[{idx}/{len(indicator_codes)}] downloading {indicator_code}")
    rows = []
    total_pages = None
    page = 1

    try:
        while total_pages is None or page <= total_pages:
            params = [("indicator", indicator_code), ("page", str(page)), ("pageSize", str(PAGE_SIZE))]
            params.extend(area_code_params(area_codes))
            payload = get_json(f"{API_BASE}/Indicator/Data", params=params)

            if total_pages is None:
                total_pages = int(payload.get("totalPages", 0) or 0)
                total_elements = int(payload.get("totalElements", 0) or 0)
                print(f"  totalElements={total_elements:,}, totalPages={total_pages}")

            for rec in payload.get("data", []):
                rows.append(flatten_record(rec))

            if total_pages == 0:
                break
            page += 1
            time.sleep(SLEEP_SECONDS)

        df = pd.DataFrame(rows)
        if not df.empty:
            df["download_indicator"] = indicator_code
            df["geoAreaCode"] = df["geoAreaCode"].astype(str)
            df = df.merge(
                iso[["GeoAreaCode_str", "iso3c", "GeoAreaName_local", "dataset"]],
                left_on="geoAreaCode",
                right_on="GeoAreaCode_str",
                how="left",
            )

        df.to_csv(out_path, index=False)
        log_rows.append({
            "indicator": indicator_code,
            "status": "downloaded",
            "rows": len(df),
            "file": str(out_path),
            "downloaded_at": datetime.now().isoformat(timespec="seconds"),
        })
        print(f"  saved rows={len(df):,}: {out_path}")
    except Exception as exc:
        error_rows.append({
            "indicator": indicator_code,
            "status": "error",
            "error": str(exc),
            "file": str(out_path),
            "downloaded_at": datetime.now().isoformat(timespec="seconds"),
        })
        print(f"  ERROR {indicator_code}: {exc}")

log = pd.DataFrame(log_rows)
errors = pd.DataFrame(error_rows)
log_path = OUT_DIR / "sdg_api_download_log.csv"
errors_path = OUT_DIR / "sdg_api_download_errors.csv"
log.to_csv(log_path, index=False)
errors.to_csv(errors_path, index=False)

print("Saved log:", log_path)
print("Saved errors:", errors_path)
print("Downloaded/skipped:", len(log), "Errors:", len(errors))

In [ ]:
# (8) Combine per-indicator files into one long CSV
indicator_files = sorted(BY_INDICATOR_DIR.glob("*.csv"))
frames = []
for path in indicator_files:
    try:
        df = pd.read_csv(path, low_memory=False)
        if not df.empty:
            frames.append(df)
    except Exception as exc:
        print("Could not read", path, exc)

if frames:
    sdg_all = pd.concat(frames, ignore_index=True, sort=False)
else:
    sdg_all = pd.DataFrame()

combined_path = OUT_DIR / "sdg_iso36_all_indicators_all_years_long.csv"
sdg_all.to_csv(combined_path, index=False)

print("Combined rows:", len(sdg_all))
print("Combined columns:", len(sdg_all.columns))
print("Saved:", combined_path)
sdg_all.head()

In [ ]:
# (9) Basic coverage summaries
if sdg_all.empty:
    print("sdg_all is empty; run the download cells first.")
else:
    summary_country = (
        sdg_all.groupby(["geoAreaCode", "geoAreaName", "iso3c"], dropna=False)
        .agg(
            rows=("value", "size"),
            indicators=("indicator", "nunique"),
            series=("series", "nunique"),
            min_year=("timePeriodStart", "min"),
            max_year=("timePeriodStart", "max"),
        )
        .reset_index()
        .sort_values(["iso3c", "geoAreaName"])
    )
    summary_indicator = (
        sdg_all.groupby(["indicator"], dropna=False)
        .agg(
            rows=("value", "size"),
            countries=("geoAreaCode", "nunique"),
            series=("series", "nunique"),
            min_year=("timePeriodStart", "min"),
            max_year=("timePeriodStart", "max"),
        )
        .reset_index()
        .sort_values("indicator")
    )

    country_summary_path = OUT_DIR / "sdg_iso36_country_coverage_summary.csv"
    indicator_summary_path = OUT_DIR / "sdg_iso36_indicator_coverage_summary.csv"
    summary_country.to_csv(country_summary_path, index=False)
    summary_indicator.to_csv(indicator_summary_path, index=False)

    print("Saved:", country_summary_path)
    print("Saved:", indicator_summary_path)
    display(summary_country.head())
    display(summary_indicator.head())

## Extract Goal 3Values at Each Country's Last Survey Year

This section creates a country-level wide table for SDG Goal 3  indicators at each country-specific survey year from `results/model_prep/last_survey_year_by_country.csv`. If no value exists exactly in the survey year, it uses the mean of nearby years within `NEARBY_YEAR_WINDOW`; if that is still unavailable, it uses the nearest available year mean as a fallback.

In [ ]:
# (10) Extract one SDG3 value per indicator-series for each country-specific survey year
LAST_SURVEY_YEAR_PATH = PROJECT_ROOT / "results" / "model_prep" / "last_survey_year_by_country.csv"
SDG_LONG_PATH = OUT_DIR / "sdg_iso36_all_indicators_all_years_long.csv"
SDG_YEAR_PATH = OUT_DIR / "sdg_year.csv"
SDG_YEAR_LONG_PATH = OUT_DIR / "sdg_year_long.csv"
SDG_YEAR_FEATURES_PATH = OUT_DIR / "sdg_year_feature_crosswalk.csv"

GOALS_TO_KEEP = {"3"}
NEARBY_YEAR_WINDOW = 2

# Aggregate/total codes used by the UN SDG API. We use these to avoid sex-, city-,
# location-, quantile-, and other subgroup-specific variables when a total exists.
AGGREGATE_DIM_CODES = {
    "dim_Reporting_Type": {"G", "_T", "ALL"},
    "dim_Quantile": {"_T", "ALL"},
    "dim_Age": {"ALLAGE", "ALL", "_T"},
    "dim_Location": {"ALLAREA", "ALL", "_T"},
    "dim_Sex": {"BOTHSEX", "ALL", "_T"},
    "dim_Disability_status": {"_T", "ALL"},
    "dim_Cities": {"NOCITI", "ALL", "_T"},
    "dim_Level_of_government": {"_T", "ALL"},
    "dim_Type_of_household": {"_T", "ALL"},
}

if not SDG_LONG_PATH.exists():
    raise FileNotFoundError(f"Missing combined SDG file: {SDG_LONG_PATH}. Run cells (7)-(8) first.")

last_year = pd.read_csv(LAST_SURVEY_YEAR_PATH)
last_year = last_year.rename(columns={"country": "GeoAreaName_local"})
last_year["GeoAreaName_local"] = last_year["GeoAreaName_local"].astype(str).str.strip()
last_year["last_survey_year"] = pd.to_numeric(last_year["last_survey_year"], errors="coerce")

country_year = (
    iso[["GeoAreaCode_str", "iso3c", "GeoAreaName_local", "dataset"]]
    .merge(last_year[["GeoAreaName_local", "last_survey_year"]], on="GeoAreaName_local", how="left")
)
missing_year = country_year.loc[country_year["last_survey_year"].isna(), "GeoAreaName_local"].tolist()
if missing_year:
    print("Countries in iso36 without last_survey_year:", missing_year)
country_year = country_year.dropna(subset=["last_survey_year"]).copy()
country_year["target_year"] = country_year["last_survey_year"].round().astype(int)

sdg = pd.read_csv(SDG_LONG_PATH, low_memory=False)
sdg["goal"] = sdg["goal"].astype(str)
sdg["indicator"] = sdg["indicator"].astype(str)
sdg["geoAreaCode"] = sdg["geoAreaCode"].astype(str)
sdg["timePeriodStart"] = pd.to_numeric(sdg["timePeriodStart"], errors="coerce")
sdg["value_num"] = pd.to_numeric(sdg["value"], errors="coerce")

sdg = sdg.loc[
    sdg["goal"].isin(GOALS_TO_KEEP)
    & sdg["geoAreaCode"].isin(country_year["GeoAreaCode_str"])
    & sdg["timePeriodStart"].notna()
    & sdg["value_num"].notna()
].copy()
sdg["year"] = sdg["timePeriodStart"].round().astype(int)

dim_cols = sorted([c for c in sdg.columns if c.startswith("dim_")])
for col in dim_cols:
    sdg[col] = sdg[col].fillna("ALL").astype(str)

def aggregate_dimension_score(df: pd.DataFrame) -> pd.Series:
    score = pd.Series(0, index=df.index, dtype=int)
    for col in dim_cols:
        allowed = AGGREGATE_DIM_CODES.get(col, {"ALL", "_T"})
        score += (~df[col].isin(allowed)).astype(int)
    return score

sdg["nonaggregate_dim_count"] = aggregate_dimension_score(sdg)
min_score = sdg.groupby(["geoAreaCode", "year", "indicator", "series"])["nonaggregate_dim_count"].transform("min")
sdg_preferred = sdg.loc[sdg["nonaggregate_dim_count"] == min_score].copy()

annual = (
    sdg_preferred.groupby(["geoAreaCode", "year", "indicator", "series"], dropna=False)
    .agg(
        value=("value_num", "mean"),
        n_source_rows=("value_num", "size"),
        n_dimension_rows=("nonaggregate_dim_count", "size"),
        selected_nonaggregate_dim_count=("nonaggregate_dim_count", "min"),
        goal=("goal", "first"),
        seriesDescription=("seriesDescription", "first"),
        attr_Units=("attr_Units", lambda x: "|".join(sorted(set(x.dropna().astype(str))))),
    )
    .reset_index()
)
annual["feature_id"] = (
    "sdg"
    + annual["goal"].astype(str)
    + "_"
    + annual["indicator"].astype(str).str.replace(".", "_", regex=False)
    + "_"
    + annual["series"].astype(str).str.replace(r"[^0-9A-Za-z_]+", "_", regex=True).str.strip("_")
)
annual["feature_label"] = annual["indicator"].astype(str) + " | " + annual["series"].astype(str) + " | " + annual["seriesDescription"].fillna("").astype(str)

features = (
    annual[["feature_id", "goal", "indicator", "series", "seriesDescription", "attr_Units"]]
    .drop_duplicates("feature_id")
    .sort_values(["goal", "indicator", "series"])
)
features.to_csv(SDG_YEAR_FEATURES_PATH, index=False)

records = []
for _, cy in country_year.iterrows():
    code = cy["GeoAreaCode_str"]
    target_year = int(cy["target_year"])
    country_data = annual.loc[annual["geoAreaCode"] == code].copy()

    for feature_id, group in country_data.groupby("feature_id", sort=False):
        exact = group.loc[group["year"] == target_year]
        if not exact.empty:
            use = exact
            method = "exact_year"
        else:
            nearby = group.loc[group["year"].between(target_year - NEARBY_YEAR_WINDOW, target_year + NEARBY_YEAR_WINDOW)]
            if not nearby.empty:
                use = nearby
                method = f"nearby_mean_pm{NEARBY_YEAR_WINDOW}"
            else:
                dist = (group["year"] - target_year).abs()
                use = group.loc[dist == dist.min()]
                method = "nearest_year_mean"

        records.append({
            "GeoAreaCode": code,
            "iso3c": cy["iso3c"],
            "country": cy["GeoAreaName_local"],
            "dataset": cy["dataset"],
            "last_survey_year": cy["last_survey_year"],
            "target_year": target_year,
            "feature_id": feature_id,
            "indicator": use["indicator"].iloc[0],
            "goal": use["goal"].iloc[0],
            "value": use["value"].mean(),
            "estimate_method": method,
            "source_years": "|".join(str(y) for y in sorted(use["year"].unique())),
            "source_n_years": use["year"].nunique(),
            "source_n_rows": use["n_source_rows"].sum(),
            "series": use["series"].iloc[0],
            "n_dimension_rows": use["n_dimension_rows"].sum(),
            "selected_nonaggregate_dim_count": use["selected_nonaggregate_dim_count"].min(),
        })

sdg_year_long = pd.DataFrame(records)
sdg_year_long = sdg_year_long.merge(features, on=["feature_id", "goal", "indicator", "series"], how="left")
sdg_year_long.to_csv(SDG_YEAR_LONG_PATH, index=False)

id_cols = ["GeoAreaCode", "iso3c", "country", "dataset", "last_survey_year", "target_year"]
sdg_year = sdg_year_long.pivot_table(index=id_cols, columns="feature_id", values="value", aggfunc="mean").reset_index()
sdg_year.columns.name = None
sdg_year = sdg_year.sort_values("country")
sdg_year.to_csv(SDG_YEAR_PATH, index=False)

print("SDG Goal 3 source rows:", len(sdg))
print("Preferred aggregate rows:", len(sdg_preferred))
print("Indicator-series features:", features.shape[0])
print("Long rows:", sdg_year_long.shape[0])
print("Wide shape:", sdg_year.shape)
print("Estimate methods:")
print(sdg_year_long["estimate_method"].value_counts())
print("Saved:", SDG_YEAR_PATH)
print("Saved:", SDG_YEAR_LONG_PATH)
print("Saved:", SDG_YEAR_FEATURES_PATH)
sdg_year.head()

## Extract All SDG Goals at Each Country's Last Survey Year

This cell creates all-goal country-level SDG files at each country's last survey year. Missing exact-year values are imputed using the mean within `NEARBY_YEAR_WINDOW = 5`; if no nearby value exists, the nearest available year mean is used.

In [ ]:
# (11) Extract one value per indicator-series for all SDG Goals at each country-specific survey year
LAST_SURVEY_YEAR_PATH = PROJECT_ROOT / "results" / "model_prep" / "last_survey_year_by_country.csv"
SDG_LONG_PATH = OUT_DIR / "sdg_iso36_all_indicators_all_years_long.csv"

# Update the standard sdg_year outputs to all-goal versions, and also save explicit all-goal copies.
SDG_YEAR_PATH = OUT_DIR / "sdg_year.csv"
SDG_YEAR_LONG_PATH = OUT_DIR / "sdg_year_long.csv"
SDG_YEAR_FEATURES_PATH = OUT_DIR / "sdg_year_feature_crosswalk.csv"
SDG_YEAR_ALL_GOALS_PATH = OUT_DIR / "sdg_year_all_goals.csv"
SDG_YEAR_ALL_GOALS_LONG_PATH = OUT_DIR / "sdg_year_all_goals_long.csv"
SDG_YEAR_ALL_GOALS_FEATURES_PATH = OUT_DIR / "sdg_year_all_goals_feature_crosswalk.csv"

NEARBY_YEAR_WINDOW = 5

# Aggregate/total codes used by the UN SDG API. We use these to avoid sex-, city-,
# location-, quantile-, and other subgroup-specific variables when a total exists.
AGGREGATE_DIM_CODES = {
    "dim_Reporting_Type": {"G", "_T", "ALL"},
    "dim_Quantile": {"_T", "ALL"},
    "dim_Age": {"ALLAGE", "ALL", "_T"},
    "dim_Location": {"ALLAREA", "ALL", "_T"},
    "dim_Sex": {"BOTHSEX", "ALL", "_T"},
    "dim_Disability_status": {"_T", "ALL"},
    "dim_Cities": {"NOCITI", "ALL", "_T"},
    "dim_Level_of_government": {"_T", "ALL"},
    "dim_Type_of_household": {"_T", "ALL"},
}

if not SDG_LONG_PATH.exists():
    raise FileNotFoundError(f"Missing combined SDG file: {SDG_LONG_PATH}. Run cells (7)-(8) first.")

last_year = pd.read_csv(LAST_SURVEY_YEAR_PATH)
last_year = last_year.rename(columns={"country": "GeoAreaName_local"})
last_year["GeoAreaName_local"] = last_year["GeoAreaName_local"].astype(str).str.strip()
last_year["last_survey_year"] = pd.to_numeric(last_year["last_survey_year"], errors="coerce")

country_year = (
    iso[["GeoAreaCode_str", "iso3c", "GeoAreaName_local", "dataset"]]
    .merge(last_year[["GeoAreaName_local", "last_survey_year"]], on="GeoAreaName_local", how="left")
)
missing_year = country_year.loc[country_year["last_survey_year"].isna(), "GeoAreaName_local"].tolist()
if missing_year:
    print("Countries in iso36 without last_survey_year:", missing_year)
country_year = country_year.dropna(subset=["last_survey_year"]).copy()
country_year["target_year"] = country_year["last_survey_year"].round().astype(int)

sdg = pd.read_csv(SDG_LONG_PATH, low_memory=False)
sdg["goal"] = sdg["goal"].astype(str)
sdg["indicator"] = sdg["indicator"].astype(str)
sdg["geoAreaCode"] = sdg["geoAreaCode"].astype(str)
sdg["timePeriodStart"] = pd.to_numeric(sdg["timePeriodStart"], errors="coerce")
sdg["value_num"] = pd.to_numeric(sdg["value"], errors="coerce")

sdg = sdg.loc[
    sdg["geoAreaCode"].isin(country_year["GeoAreaCode_str"])
    & sdg["timePeriodStart"].notna()
    & sdg["value_num"].notna()
].copy()
sdg["year"] = sdg["timePeriodStart"].round().astype(int)

dim_cols = sorted([c for c in sdg.columns if c.startswith("dim_")])
for col in dim_cols:
    sdg[col] = sdg[col].fillna("ALL").astype(str)

# Prefer total/aggregate rows. Within each country-year-indicator-series, keep rows with the
# smallest number of non-aggregate dimensions, then average remaining rows to one value.
def aggregate_dimension_score(df: pd.DataFrame) -> pd.Series:
    score = pd.Series(0, index=df.index, dtype=int)
    for col in dim_cols:
        allowed = AGGREGATE_DIM_CODES.get(col, {"ALL", "_T"})
        score += (~df[col].isin(allowed)).astype(int)
    return score

sdg["nonaggregate_dim_count"] = aggregate_dimension_score(sdg)
min_score = sdg.groupby(["geoAreaCode", "year", "indicator", "series"])["nonaggregate_dim_count"].transform("min")
sdg_preferred = sdg.loc[sdg["nonaggregate_dim_count"] == min_score].copy()

annual = (
    sdg_preferred.groupby(["geoAreaCode", "year", "indicator", "series"], dropna=False)
    .agg(
        value=("value_num", "mean"),
        n_source_rows=("value_num", "size"),
        n_dimension_rows=("nonaggregate_dim_count", "size"),
        selected_nonaggregate_dim_count=("nonaggregate_dim_count", "min"),
        goal=("goal", "first"),
        seriesDescription=("seriesDescription", "first"),
        attr_Units=("attr_Units", lambda x: "|".join(sorted(set(x.dropna().astype(str))))),
    )
    .reset_index()
)
annual["feature_id"] = (
    "sdg"
    + annual["goal"].astype(str)
    + "_"
    + annual["indicator"].astype(str).str.replace(".", "_", regex=False)
    + "_"
    + annual["series"].astype(str).str.replace(r"[^0-9A-Za-z_]+", "_", regex=True).str.strip("_")
)
annual["feature_label"] = annual["indicator"].astype(str) + " | " + annual["series"].astype(str) + " | " + annual["seriesDescription"].fillna("").astype(str)

features = (
    annual[["feature_id", "goal", "indicator", "series", "seriesDescription", "attr_Units"]]
    .drop_duplicates("feature_id")
    .sort_values(["goal", "indicator", "series"])
)

records = []
for _, cy in country_year.iterrows():
    code = cy["GeoAreaCode_str"]
    target_year = int(cy["target_year"])
    country_data = annual.loc[annual["geoAreaCode"] == code].copy()

    for feature_id, group in country_data.groupby("feature_id", sort=False):
        exact = group.loc[group["year"] == target_year]
        if not exact.empty:
            use = exact
            method = "exact_year"
        else:
            nearby = group.loc[group["year"].between(target_year - NEARBY_YEAR_WINDOW, target_year + NEARBY_YEAR_WINDOW)]
            if not nearby.empty:
                use = nearby
                method = f"nearby_mean_pm{NEARBY_YEAR_WINDOW}"
            else:
                dist = (group["year"] - target_year).abs()
                use = group.loc[dist == dist.min()]
                method = "nearest_year_mean"

        records.append({
            "GeoAreaCode": code,
            "iso3c": cy["iso3c"],
            "country": cy["GeoAreaName_local"],
            "dataset": cy["dataset"],
            "last_survey_year": cy["last_survey_year"],
            "target_year": target_year,
            "feature_id": feature_id,
            "indicator": use["indicator"].iloc[0],
            "goal": use["goal"].iloc[0],
            "value": use["value"].mean(),
            "estimate_method": method,
            "source_years": "|".join(str(y) for y in sorted(use["year"].unique())),
            "source_n_years": use["year"].nunique(),
            "source_n_rows": use["n_source_rows"].sum(),
            "series": use["series"].iloc[0],
            "n_dimension_rows": use["n_dimension_rows"].sum(),
            "selected_nonaggregate_dim_count": use["selected_nonaggregate_dim_count"].min(),
        })

sdg_year_long = pd.DataFrame(records)
sdg_year_long = sdg_year_long.merge(features, on=["feature_id", "goal", "indicator", "series"], how="left")

id_cols = ["GeoAreaCode", "iso3c", "country", "dataset", "last_survey_year", "target_year"]
sdg_year = sdg_year_long.pivot_table(index=id_cols, columns="feature_id", values="value", aggfunc="mean").reset_index()
sdg_year.columns.name = None
sdg_year = sdg_year.sort_values("country")

# Save standard names and explicit all-goal names.
sdg_year.to_csv(SDG_YEAR_PATH, index=False)
sdg_year_long.to_csv(SDG_YEAR_LONG_PATH, index=False)
features.to_csv(SDG_YEAR_FEATURES_PATH, index=False)
sdg_year.to_csv(SDG_YEAR_ALL_GOALS_PATH, index=False)
sdg_year_long.to_csv(SDG_YEAR_ALL_GOALS_LONG_PATH, index=False)
features.to_csv(SDG_YEAR_ALL_GOALS_FEATURES_PATH, index=False)

print("All-goal source rows:", len(sdg))
print("Preferred aggregate rows:", len(sdg_preferred))
print("Indicator-series features:", features.shape[0])
print("Long rows:", sdg_year_long.shape[0])
print("Wide shape:", sdg_year.shape)
def goal_sort_key(x):
    x = str(x)
    return (0, int(float(x))) if x.replace('.', '', 1).isdigit() else (1, x)

print("Goals:", ", ".join(sorted(features["goal"].astype(str).unique(), key=goal_sort_key)))
print("Estimate methods:")
print(sdg_year_long["estimate_method"].value_counts())
print("Saved:", SDG_YEAR_PATH)
print("Saved:", SDG_YEAR_ALL_GOALS_PATH)
sdg_year.head()
